# Hybrid Robotic Vision: Notebook 3 — ROI Still Compression with JPEG

This notebook implements the **JPEG version** of Notebook 3 for the hybrid visual telemetry experiment.

Its job is to simulate the **still-image side channel** of the hybrid system:

1. Read the ROI manifest generated in Notebook 2  
2. Reconstruct the actual ROI crops from:
   - the **original source video**
   - the **encoded/base video**
3. Compress the original-source ROI crops as **JPEG still images**
4. Save a decoded still artifact for downstream analysis
5. Measure per-ROI bitrate / file size
6. Save a clean still-image manifest for Notebook 4

## Why this version uses JPEG

This Colab-ready version uses **JPEG** because it is easy to run reliably in Colab without external codec tools.

That lets us:
- complete the full E2E experimental pipeline
- measure ROI-side bitrate cost
- compare semantic quality between base-video crops and transmitted stills

Later, the same structure can be reused for:
- JPEG XL
- JPEG AI
- other still-image codecs

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define project paths and select the sequence

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
DATA = ROOT / "data" / "uavdt"
VIDEOS = DATA / "videos"
RUNS = ROOT / "runs"

SEQUENCE_NAME = "M0101"   # <-- EDIT THIS
REGIME = "low"            # <-- choose: "low" or "moderate"

ORIGINAL_VIDEO = VIDEOS / f"{SEQUENCE_NAME}.mp4"

ROI_MANIFEST_DIR = RUNS / "yolo_roi_manifest" / SEQUENCE_NAME / REGIME / "manifests"
ROI_MANIFEST_CSV = ROI_MANIFEST_DIR / "roi_manifest.csv"

if REGIME == "low":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_low" / "video_only" / SEQUENCE_NAME
elif REGIME == "moderate":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_moderate" / "video_only" / SEQUENCE_NAME
else:
    raise ValueError("REGIME must be 'low' or 'moderate'")

encoded_video_candidates = [p for p in ENCODED_VIDEO_DIR.iterdir() if p.suffix.lower() in [".mp4", ".mkv", ".mov"]]
if len(encoded_video_candidates) == 0:
    raise FileNotFoundError(f"No encoded video found in {ENCODED_VIDEO_DIR}")
ENCODED_VIDEO = encoded_video_candidates[0]

OUT_DIR = RUNS / "roi_still_compression_jpeg" / SEQUENCE_NAME / REGIME
ORIG_CROP_DIR = OUT_DIR / "original_crops"
COMP_CROP_DIR = OUT_DIR / "compressed_video_crops"
JPEG_DIR = OUT_DIR / "jpeg"
DECODED_JPEG_DIR = OUT_DIR / "decoded_jpeg"
MANIFEST_DIR = OUT_DIR / "manifests"

for p in [ORIG_CROP_DIR, COMP_CROP_DIR, JPEG_DIR, DECODED_JPEG_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Original video:", ORIGINAL_VIDEO)
print("Encoded video:", ENCODED_VIDEO)
print("ROI manifest:", ROI_MANIFEST_CSV)
print("Output dir:", OUT_DIR)

## 3. Install packages

In [ ]:
!pip install -q opencv-python-headless pandas tqdm matplotlib pillow
!apt-get update -qq
!apt-get install -y ffmpeg

## 4. Imports

In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from pathlib import Path

## 5. Load the ROI manifest

In [ ]:
if not ROI_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"ROI manifest not found: {ROI_MANIFEST_CSV}")

roi_manifest_df = pd.read_csv(ROI_MANIFEST_CSV)
print(f"Loaded ROI manifest with {len(roi_manifest_df)} rows")
roi_manifest_df.head()

## 6. Configure crop and compression settings

In [ ]:
OVERRIDE_PADDING_FRAC = None
MAX_ROIS = None   # Example: 100

STILL_CODEC = "jpeg"
JPEG_QUALITY = 65
JPEG_OPTIMIZE = True
DECODE_STILL = True

print("OVERRIDE_PADDING_FRAC =", OVERRIDE_PADDING_FRAC)
print("MAX_ROIS =", MAX_ROIS)
print("STILL_CODEC =", STILL_CODEC)
print("JPEG_QUALITY =", JPEG_QUALITY)
print("JPEG_OPTIMIZE =", JPEG_OPTIMIZE)
print("DECODE_STILL =", DECODE_STILL)

## 7. Helper functions

In [ ]:
def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return frame

def padded_box(x1, y1, x2, y2, width, height, pad_frac=0.15):
    w = x2 - x1
    h = y2 - y1
    px = w * pad_frac
    py = h * pad_frac
    nx1 = max(0, int(round(x1 - px)))
    ny1 = max(0, int(round(y1 - py)))
    nx2 = min(width - 1, int(round(x2 + px)))
    ny2 = min(height - 1, int(round(y2 + py)))
    return nx1, ny1, nx2, ny2

def file_size_bytes(path):
    path = Path(path)
    return path.stat().st_size if path.exists() else None

## 8. Filter the manifest for the current run

In [ ]:
work_df = roi_manifest_df.copy()
if MAX_ROIS is not None:
    work_df = work_df.head(MAX_ROIS).copy()

print(f"Using {len(work_df)} ROI rows in this run")
work_df.head()

## 9. Extract the actual crops from original and encoded videos

In [ ]:
crop_rows = []

for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Extracting crops"):
    enc_frame_idx = int(row["enc_frame_idx"])
    orig_frame_idx = int(row["orig_frame_idx"])

    orig_frame = read_frame(ORIGINAL_VIDEO, orig_frame_idx)
    comp_frame = read_frame(ENCODED_VIDEO, enc_frame_idx)

    if orig_frame is None or comp_frame is None:
        continue

    oh, ow = orig_frame.shape[:2]
    ch, cw = comp_frame.shape[:2]

    pad_frac = float(row["padding_frac"]) if OVERRIDE_PADDING_FRAC is None else float(OVERRIDE_PADDING_FRAC)

    ox1, oy1, ox2, oy2 = padded_box(
        float(row["orig_x1"]), float(row["orig_y1"]),
        float(row["orig_x2"]), float(row["orig_y2"]),
        ow, oh, pad_frac=pad_frac
    )

    cx1, cy1, cx2, cy2 = padded_box(
        float(row["enc_x1"]), float(row["enc_y1"]),
        float(row["enc_x2"]), float(row["enc_y2"]),
        cw, ch, pad_frac=pad_frac
    )

    orig_crop = orig_frame[oy1:oy2, ox1:ox2]
    comp_crop = comp_frame[cy1:cy2, cx1:cx2]

    if orig_crop.size == 0 or comp_crop.size == 0:
        continue

    crop_base = (
        f'encframe{enc_frame_idx:06d}_'
        f'origframe{orig_frame_idx:06d}_'
        f'det{int(row["det_idx"]):05d}_'
        f'{row["cls_name"]}'
    )

    orig_crop_path = ORIG_CROP_DIR / f"{crop_base}.png"
    comp_crop_path = COMP_CROP_DIR / f"{crop_base}.png"

    cv2.imwrite(str(orig_crop_path), orig_crop)
    cv2.imwrite(str(comp_crop_path), comp_crop)

    crop_rows.append({
        **row.to_dict(),
        "crop_base": crop_base,
        "effective_padding_frac": pad_frac,
        "orig_crop_path": str(orig_crop_path),
        "comp_crop_path": str(comp_crop_path),
        "orig_crop_width": int(orig_crop.shape[1]),
        "orig_crop_height": int(orig_crop.shape[0]),
        "comp_crop_width": int(comp_crop.shape[1]),
        "comp_crop_height": int(comp_crop.shape[0]),
        "orig_crop_png_size_bytes": file_size_bytes(orig_crop_path),
        "comp_crop_png_size_bytes": file_size_bytes(comp_crop_path),
    })

crop_df = pd.DataFrame(crop_rows)
print(f"Extracted {len(crop_df)} crop pairs")
crop_df.head()

## 10. Compress original crops as JPEG stills

In [ ]:
compression_rows = []

if len(crop_df) == 0:
    print("No crop rows available for compression.")
else:
    for _, row in tqdm(crop_df.iterrows(), total=len(crop_df), desc="Compressing JPEG stills"):
        orig_crop_path = Path(row["orig_crop_path"])
        still_path = JPEG_DIR / f"{Path(row['crop_base']).name}.jpg"
        decoded_still_path = DECODED_JPEG_DIR / f"{Path(row['crop_base']).name}.png"

        still_ok = False
        decode_ok = False

        try:
            img = Image.open(orig_crop_path).convert("RGB")
            img.save(still_path, format="JPEG", quality=JPEG_QUALITY, optimize=JPEG_OPTIMIZE)
            still_ok = still_path.exists()

            if still_ok and DECODE_STILL:
                img2 = Image.open(still_path).convert("RGB")
                img2.save(decoded_still_path, format="PNG")
                decode_ok = decoded_still_path.exists()

        except Exception:
            still_ok = False
            decode_ok = False

        compression_rows.append({
            **row.to_dict(),
            "still_codec": STILL_CODEC,
            "still_quality": JPEG_QUALITY,
            "still_path": str(still_path) if still_ok else None,
            "still_size_bytes": file_size_bytes(still_path) if still_ok else None,
            "decoded_still_path": str(decoded_still_path) if decode_ok else None,
            "decoded_still_size_bytes": file_size_bytes(decoded_still_path) if decode_ok else None,
            "still_compression_ok": bool(still_ok),
            "still_decode_ok": bool(decode_ok),
        })

roi_still_df = pd.DataFrame(compression_rows)
print(f"Prepared ROI still manifest with {len(roi_still_df)} rows")
roi_still_df.head()

## 11. Save the still-image manifest

In [ ]:
roi_still_manifest_csv = MANIFEST_DIR / "roi_still_manifest.csv"
roi_still_df.to_csv(roi_still_manifest_csv, index=False)

print("Saved ROI still manifest to:")
print(roi_still_manifest_csv)

## 12. Quick compression summary

In [ ]:
if len(roi_still_df) == 0:
    print("No ROI still rows available.")
else:
    size_series = roi_still_df["still_size_bytes"].dropna()
    summary = {
        "still_codec": STILL_CODEC,
        "num_roi_rows": int(len(roi_still_df)),
        "num_successful_still_compressions": int(roi_still_df["still_compression_ok"].sum()),
        "num_successful_still_decodes": int(roi_still_df["still_decode_ok"].sum()),
        "total_still_bytes": int(size_series.sum()) if len(size_series) > 0 else 0,
        "mean_still_bytes": float(size_series.mean()) if len(size_series) > 0 else None,
        "median_still_bytes": float(size_series.median()) if len(size_series) > 0 else None,
    }
    print(json.dumps(summary, indent=2))

## 13. Visual sanity check of crop pairs and decoded JPEG stills

In [ ]:
import matplotlib.pyplot as plt

if len(roi_still_df) == 0:
    print("No rows available for visualization.")
else:
    vis_df = roi_still_df.head(min(4, len(roi_still_df)))

    n_rows = len(vis_df)
    n_cols = 3
    plt.figure(figsize=(12, 3 * n_rows))

    for i, (_, row) in enumerate(vis_df.iterrows(), 1):
        orig_img = np.array(Image.open(row["orig_crop_path"]))
        comp_img = np.array(Image.open(row["comp_crop_path"]))

        plt.subplot(n_rows, n_cols, (i - 1) * n_cols + 1)
        plt.imshow(orig_img)
        plt.title(f"Original crop\n{row['cls_name']}")
        plt.axis("off")

        plt.subplot(n_rows, n_cols, (i - 1) * n_cols + 2)
        plt.imshow(comp_img)
        plt.title("Compressed-video crop")
        plt.axis("off")

        plt.subplot(n_rows, n_cols, (i - 1) * n_cols + 3)
        if isinstance(row.get("decoded_still_path", None), str) and Path(row["decoded_still_path"]).exists():
            dec_img = np.array(Image.open(row["decoded_still_path"]))
            plt.imshow(dec_img)
            plt.title("Decoded JPEG crop")
        else:
            blank = np.zeros((max(1, orig_img.shape[0]), max(1, orig_img.shape[1]), 3), dtype=np.uint8)
            plt.imshow(blank)
            plt.title("Decoded JPEG crop\n(not available)")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 14. Save a run summary

In [ ]:
size_series = roi_still_df["still_size_bytes"].dropna() if len(roi_still_df) > 0 else pd.Series(dtype=float)

run_summary = {
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "original_video": str(ORIGINAL_VIDEO),
    "encoded_video": str(ENCODED_VIDEO),
    "roi_manifest_csv": str(ROI_MANIFEST_CSV),
    "roi_still_manifest_csv": str(roi_still_manifest_csv),
    "num_input_roi_rows": int(len(work_df)),
    "num_extracted_crop_rows": int(len(crop_df)),
    "num_output_rows": int(len(roi_still_df)),
    "override_padding_frac": OVERRIDE_PADDING_FRAC,
    "still_codec": STILL_CODEC,
    "jpeg_quality": JPEG_QUALITY,
    "decode_still": DECODE_STILL,
    "total_still_bytes": int(size_series.sum()) if len(size_series) > 0 else 0,
    "num_successful_still_compressions": int(roi_still_df["still_compression_ok"].sum()) if len(roi_still_df) > 0 else 0,
    "num_successful_still_decodes": int(roi_still_df["still_decode_ok"].sum()) if len(roi_still_df) > 0 else 0,
}

run_summary_path = MANIFEST_DIR / "run_summary.json"
with open(run_summary_path, "w") as f:
    json.dump(run_summary, f, indent=2)

print("Saved run summary to:")
print(run_summary_path)
print(json.dumps(run_summary, indent=2))

## 15. What this notebook completed

At this point you now have:
- reconstructed original-source ROI crops
- reconstructed compressed-video ROI crops
- JPEG still files for the original crops
- decoded JPEG crops for downstream analysis
- a complete per-ROI still-image manifest
- the first direct measurement of ROI-side bitrate cost

## What Notebook 4 should do next

Notebook 4 should read `roi_still_manifest.csv` and:
1. run classification on compressed-video crops
2. run classification on decoded JPEG stills
3. compare confidence and class predictions
4. aggregate results against total video + ROI bitrate